# 創薬仮説 バッチ生成ノートブック
**Batch Drug Discovery Hypothesis Generator**

1つの疾患に対して複数の遺伝子を順番に処理し、レポートを自動生成します。

- **Step 1**: LLM・言語設定
- **Step 2**: 疾患を選択
- **Step 3**: 対象遺伝子リストを入力
- **Step 4**: バッチ実行（全遺伝子を自動処理）

## Step 1: LLM・言語設定

In [ ]:
import sys
sys.path.insert(0, '.')

for mod in list(sys.modules.keys()):
    if 'llm' in mod:
        del sys.modules[mod]

from llm.ollama_client import OllamaClient

MODEL = 'qwen2.5:14b'   # 例: 'llama3.1', 'qwen2.5:14b'
LANG  = 'en'            # 'ja' = 日本語 / 'en' = English

llm = OllamaClient(model=MODEL)
if llm.is_available():
    print(f'✓ Ollama ({MODEL}) 起動確認')
    print(f'  言語: {LANG}')
else:
    raise RuntimeError('⚠ Ollama に接続できません。ターミナルで ollama serve を実行してください。')

# ── コンテキスト情報量の設定 ──────────────────────────────
# 数値を変更するとLLMへ渡す情報量が変わります。
# 値を大きくすると仮説の根拠が豊富になりますが、生成時間も長くなります。

CONTEXT_CONFIG = dict(
    max_papers       = 5,    # 論文数（PubMed）
    abstract_chars   = 600,  # アブストラクト 1件あたりの文字数
    max_drugs        = 8,    # 薬剤数（ChEMBL + OpenTargets）
    max_gwas         = 5,    # GWAS ヒット数
    max_clinvar      = 5,    # ClinVar バリアント数
    max_interactions = 10,   # PPI インタラクター数（IntAct）
    max_trials       = 6,    # 臨床試験数（ClinicalTrials.gov）
    max_reactome     = 10,   # Reactome パスウェイ数
    gtex_top_n       = 5,    # GTEx 上位発現組織数
    hpa_top_n        = 8,    # Human Protein Atlas 組織数
    max_dgidb        = 8,    # DGIdb 薬剤-遺伝子相互作用数
    uniprot_chars    = 500,  # UniProt function 文字数
    uniprot_keywords = 10,   # UniProt キーワード数
    uniprot_go_terms = 8,    # UniProt GO term 数
)

# 軽量モード（バッチ速度優先）
# CONTEXT_CONFIG = dict(max_papers=2, abstract_chars=150, max_drugs=3,
#                       max_gwas=2, max_clinvar=2, max_interactions=6,
#                       max_trials=2, max_reactome=4, gtex_top_n=2,
#                       hpa_top_n=3, max_dgidb=3, uniprot_chars=150,
#                       uniprot_keywords=5, uniprot_go_terms=3)

from aggregator import DEFAULT_CONTEXT_CONFIG
print('\nコンテキスト設定:')
for k, v in CONTEXT_CONFIG.items():
    diff = f'  ← デフォルト: {DEFAULT_CONTEXT_CONFIG.get(k)}' if v != DEFAULT_CONTEXT_CONFIG.get(k) else ''
    print(f'  {k:<22} = {v}{diff}')

## Step 2: 疾患を選択

疾患名を入力して検索し、リストから選択してください。

In [ ]:
import requests
import ipywidgets as widgets
from IPython.display import display

OT_API = 'https://api.platform.opentargets.org/api/v4/graphql'

def _ot_search(keyword, entity):
    q = '''
    query ($q: String!, $e: [String!]) {
      search(queryString: $q, entityNames: $e, page: {index: 0, size: 15}) {
        hits { id name description entity }
      }
    }
    '''
    r = requests.post(OT_API, json={'query': q, 'variables': {'q': keyword, 'e': [entity]}}, timeout=15)
    r.raise_for_status()
    return [h for h in r.json()['data']['search']['hits'] if h['entity'] == entity]

selected_disease = {'id': None, 'name': None}

box  = widgets.Text(placeholder='疾患名を入力... (例: Duchenne muscular dystrophy)',
                    layout=widgets.Layout(width='440px'))
btn  = widgets.Button(description='検索', button_style='primary',
                      layout=widgets.Layout(width='70px'))
lst  = widgets.Select(options=[], rows=8,
                      layout=widgets.Layout(width='700px'))
stat = widgets.Label(value='疾患名を入力して「検索」を押してください')

def on_search(_):
    kw = box.value.strip()
    if not kw:
        stat.value = '⚠ キーワードを入力してください'; return
    stat.value = '検索中...'
    try:
        hits = _ot_search(kw, 'disease')
        if not hits:
            stat.value = f'「{kw}」に一致する疾患が見つかりませんでした'
            lst.options = []; return
        lst.options = [
            (f"{h['name']}  [{h['id']}]  {(h.get('description') or '')[:60]}", h)
            for h in hits
        ]
        stat.value = f'{len(hits)} 件見つかりました。リストから選択してください'
    except Exception as e:
        stat.value = f'エラー: {e}'

def on_select(change):
    val = change['new']
    if val:
        selected_disease['id']   = val['id']
        selected_disease['name'] = val['name']
        stat.value = f'✓ 選択済み: {val["name"]}  ({val["id"]})'

btn.on_click(on_search)
lst.observe(on_select, names='value')
display(widgets.VBox([widgets.HBox([box, btn]), lst, stat]))

## Step 3: 遺伝子リストを入力

`GENE_LIST_INPUT` を編集してセルを実行してください（1行1遺伝子 または カンマ区切り）。

In [ ]:
import re, requests
from IPython.display import display, HTML

# ── ここを編集してセルを実行 ──────────────────────────────
GENE_LIST_INPUT = """
ACACA
ACACB
"""
# ─────────────────────────────────────────────────────────

def _parse(text):
    return [g.strip().upper() for g in re.split(r'[,\n\r]+', text) if g.strip()]

def _validate_format(symbol):
    return bool(re.match(r'^[A-Z][A-Z0-9\-]{0,19}$', symbol))

def _check_hgnc(symbols):
    try:
        joined = '+OR+'.join(f'symbol:{s}' for s in symbols)
        r = requests.get(
            f'https://rest.genenames.org/search/{joined}',
            headers={'Accept': 'application/json'}, timeout=10,
        )
        found = {d['symbol'] for d in r.json().get('response', {}).get('docs', [])}
        return {s: s in found for s in symbols}
    except Exception:
        return {s: _validate_format(s) for s in symbols}

raw = _parse(GENE_LIST_INPUT)
if not raw:
    print('⚠ 遺伝子が入力されていません')
    GENE_LIST = []
else:
    fmt_ok       = [g for g in raw if _validate_format(g)]
    fmt_bad      = [g for g in raw if not _validate_format(g)]
    hgnc         = _check_hgnc(fmt_ok) if fmt_ok else {}
    confirmed    = [g for g in fmt_ok if hgnc.get(g, True)]
    unrecognized = [g for g in fmt_ok if not hgnc.get(g, True)]
    GENE_LIST    = confirmed + unrecognized

    rows = ''
    for g in confirmed:
        rows += f'<tr><td style="padding:3px 10px;color:#1a7f37">✓</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#555;font-size:12px">HGNC confirmed</td></tr>'
    for g in unrecognized:
        rows += f'<tr><td style="padding:3px 10px;color:#9a6700">△</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#9a6700;font-size:12px">未確認（処理は継続）</td></tr>'
    for g in fmt_bad:
        rows += f'<tr><td style="padding:3px 10px;color:#cf222e">✕</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#cf222e;font-size:12px">形式エラー（スキップ）</td></tr>'

    note = f'<p style="color:#cf222e;font-size:12px;margin:4px 0 0">⚠ スキップ: {", ".join(fmt_bad)}</p>' if fmt_bad else ''
    display(HTML(f'''
    <b>✓ {len(GENE_LIST)} 遺伝子を登録</b>
    <table style="margin-top:6px;border-collapse:collapse;border:1px solid #ddd;font-size:13px">
      <thead><tr style="background:#f5f5f5">
        <th style="padding:3px 10px"></th>
        <th style="padding:3px 10px;text-align:left">遺伝子</th>
        <th style="padding:3px 10px;text-align:left">ステータス</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>{note}
    '''))

## Step 4: バッチ実行

全遺伝子を順番に処理します。各遺伝子のレポートは `reports/{GENE}_{DISEASE}/` に保存されます。

> ⚠ **注意**: LLM生成は遺伝子ごとに数分かかります。遺伝子数 × 生成時間を見込んでください。

### 各遺伝子に対して収集・使用するデータ

| データソース | 収集内容 | 設定キー |
|------------|---------|---------|
| PubMed | 論文・アブストラクト | `max_papers`, `abstract_chars` |
| OpenTargets | 遺伝子×疾患関連スコア | — |
| UniProt | タンパク質機能・局在・GO term | `uniprot_chars`, `uniprot_keywords`, `uniprot_go_terms` |
| GWAS Catalog | 遺伝的関連研究 | `max_gwas` |
| ClinVar | 病的変異 | `max_clinvar` |
| ChEMBL / OpenTargets | 既存薬・フェーズ・作用機序 | `max_drugs` |
| IntAct | タンパク質相互作用 (PPI) | `max_interactions` |
| gnomAD | 集団制約スコア pLI / LOEUF | — |
| GTEx | 組織別発現量 TPM | `gtex_top_n` |
| Human Protein Atlas | タンパク質発現・細胞内局在 | `hpa_top_n` |
| DGIdb | 薬剤–遺伝子相互作用 | `max_dgidb` |
| ClinicalTrials.gov | 臨床試験 | `max_trials` |
| AlphaFold DB | 構造信頼度 pLDDT | — |
| Reactome | 生物学的パスウェイ | `max_reactome` |
| PubChem / openFDA | 毒性アッセイ・副作用報告 | — |

In [ ]:
# モジュールを最新状態で再読み込み（編集後の再実行に対応）
import sys
for m in list(sys.modules):
    if m.split('.')[0] in ('collectors', 'aggregator', 'hypothesis', 'network', 'report', 'pipeline'):
        del sys.modules[m]

from IPython.display import display, Markdown, HTML
import pipeline
import report

# ── 入力チェック ──────────────────────────────────────────
if not selected_disease.get('name'):
    raise ValueError('⚠ Step 2 で疾患を選択してください')
if not GENE_LIST:
    raise ValueError('⚠ Step 3 で遺伝子を入力してセルを実行してください')

# ── バッチ実行（収集 → PPI/エンリッチメント → 仮説 → 評価 → 保存） ──
results = pipeline.run_batch(
    GENE_LIST, selected_disease, llm,
    lang=LANG, context_config=CONTEXT_CONFIG, verbose=True,
)

# ── サマリー表示 + 保存 ────────────────────────────────────
print('\n' + '=' * 60 + '\nバッチ完了 — サマリー')
DISEASE = selected_disease['name']
display(HTML(report.summary_html(results, DISEASE)))

summary_path = pipeline.save_summary(results, DISEASE)
print(f'サマリーMD保存: {summary_path}')
display(Markdown(report.summary_md(results, DISEASE, '')))
